# 01 Parameter-Efficient Fine-Tuning: LoRA, QLoRA & Rank Decomposition

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("Phase 6 Environment Loaded.")

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.pretrained = nn.Linear(in_features, out_features, bias=False)
        self.pretrained.weight.requires_grad = False # Freeze W_0
        
        self.r = rank
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.pretrained(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

lora_layer = LoRALinear(1024, 1024, rank=8)
x = torch.randn(2, 1024)
output = lora_layer(x)
print(f"LoRA Forward Output Shape: {output.shape}")